# Look at mapping breadth and depth of 100 x metagenomes to singleclust genes

In [1]:
import polars as pl
import glob
import os
import screed
import csv
import screed

## Read in individual metag x species singleclust depth reports

These files contain depth-of-mapping information for all our genes.

In [2]:
DIR='../outputs.cds/singleclust/bam.bak'
template = '../outputs.cds/singleclust/bam/{metag}.x.{species}.depth.txt'

def read_depth_txt(metag, species, *, exclude_ends=75):
    filename = template.format(metag=metag, species=species)
    df = pl.read_csv(filename, separator='\t', has_header=False,
                 new_columns=('gene', 'pos', 'cov', 'foo')).select(['gene', 'pos', 'cov'])

    sum_df = df.group_by('gene').all().with_columns(
        # select slice [75:-75]
        (pl.col("pos").list.slice(exclude_ends, -exclude_ends).list.len()).alias("len"),
        (pl.col("cov").list.slice(exclude_ends, -exclude_ends)),
        (pl.col("cov").list.slice(exclude_ends, -exclude_ends).list.filter(pl.element() > 0)).list.len().alias("hits"),
    ).with_columns(
        (pl.lit(metag).alias("metag")),
        (pl.lit(species).alias("species")),
        # summarize: average depth across contig,
        (pl.col("cov").list.sum() / pl.col("len")).alias("depth_all"),
        # average depth across covered bases,
        (pl.col("cov").list.sum() / pl.col("hits")).alias("depth_cov"),
        # fraction of bases covered
        (pl.col("hits") / pl.col("cov").list.len()).alias("breadth"),
    ).select(["metag", "species", "gene", "len", "hits", "breadth", "depth_all", "depth_cov"])
    return sum_df

read_depth_txt('ERR1135199', 's__Cryptobacteroides sp900546925')

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""GIIIOOFK_00161""",930,0,0.0,0.0,NaN
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""ELJKHIPJ_00342""",339,222,0.654867,1.079646,1.648649
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""LELOEEPG_00778""",435,317,0.728736,1.045977,1.435331
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""FNIGJHLI_01182""",1542,1087,0.704929,1.389105,1.970561
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""CPBNAKKL_00323""",561,392,0.698752,1.135472,1.625
…,…,…,…,…,…,…,…
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""KJEHENCE_01089""",894,479,0.535794,0.536913,1.002088
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""ELKPFNCN_00187""",870,0,0.0,0.0,NaN
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""BNDPAENK_00069""",1212,1076,0.887789,2.096535,2.361524


In [3]:
# Read them all in!
filenames = glob.glob(f"{DIR}/*.depth.txt")

dflist = []
for i, n in enumerate(filenames):
    if i % 100 == 0:
        print(f"{i} of {len(filenames)}")
    n = os.path.basename(n)
    metag, _, species, _ = n.split('.', 3)
    dflist.append(read_depth_txt(metag, species))

depth_df = pl.concat(dflist)

print(f"read {len(filenames)} depth files.")

0 of 1400
100 of 1400
200 of 1400
300 of 1400
400 of 1400
500 of 1400
600 of 1400
700 of 1400
800 of 1400
900 of 1400
1000 of 1400
1100 of 1400
1200 of 1400
1300 of 1400
read 1400 depth files.


In [4]:
depth_df.filter(pl.col('depth_all').is_not_nan()).sort(by='depth_all', descending=True).filter(pl.col('depth_all') > 0.0)

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64
"""SRR14369134""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,17800.038462,17800.038462
"""SRR11489750""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,12280.894587,12280.894587
"""SRR12795790""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,8948.836182,8948.836182
"""SRR10209683""","""s__Mogibacterium_A kristiansen…","""AAOFCIJB_00277""",1119,1119,1.0,8170.679178,8170.679178
"""SRR17241663""","""s__Mogibacterium_A kristiansen…","""JMKEGFFJ_01326""",1899,1899,1.0,7469.317536,7469.317536
…,…,…,…,…,…,…,…
"""ERR3211876""","""s__Gemmiger qucibialis""","""PMKFNJDI_00477""",3072,4,0.001302,0.001302,1.0
"""SRR8655118""","""s__Cryptobacteroides sp9005469…","""CMHHCIHA_00893""",2016,2,0.000992,0.000992,1.0
"""SRR11124687""","""s__Prevotella sp002251295""","""OPHKNFIJ_00863""",2094,2,0.000955,0.000955,1.0


## Summarize our mapping breadth results across all the metagenomes

In [5]:
# require 10% of each gene to be covered by at least one read
BREADTH_CUTOFF = 0.1

In [6]:
# aggregate across all metagenomes;
# calculate fraction of metagenomes for which gene mapping exceeds our breadth cutoff
agg_df = depth_df.group_by(['species', 'gene']).agg(
    # get fraction of columns where breadth is greater than cutoff as 'f'
    ((pl.col("breadth") >= BREADTH_CUTOFF).sum() / pl.col("breadth").len()).alias("f"),
#    (pl.col("depth").filter(pl.col("depth").is_not_nan()).mean()),
)
agg_df

species,gene,f
str,str,f64
"""s__Sodaliphilus sp004557565""","""NINKMKKC_00022""",0.79
"""s__Cryptobacteroides sp9005469…","""BEKEPKDC_01232""",0.82
"""s__Sodaliphilus sp004557565""","""FLLLPNHA_00844""",0.65
"""s__Cryptobacteroides sp9005469…","""BEKEPKDC_00394""",0.84
"""s__Cryptobacteroides sp9005469…","""MFHJPMDA_00569""",0.38
…,…,…
"""s__Cryptobacteroides sp9005469…","""CPBNAKKL_00778""",0.85
"""s__Prevotella sp002251295""","""OJEBGNEI_00601""",0.84
"""s__Sodaliphilus sp004557565""","""EDCDMAMI_01817""",0.91


In [7]:
# print information out by species
for species in sorted(agg_df['species'].unique()):
    print(species)
    foo_df = agg_df.filter(pl.col("species") == species)
    foo_df = foo_df.sort(by='f', descending=True).filter(pl.col('f') > 0.8)
    print(foo_df)

    top50_names = set(foo_df.head(50)['gene'].to_list())

    outfile = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.fa'
    print(outfile)
    outfp = open(outfile, 'wt')
    for record in screed.open(f'../outputs.cds/singleclust/{species}.cds3.min50.dedup.fa'):
        name = record.name.split(' ')[0]
        if name in top50_names:
            top50_names.remove(name)
            outfp.write(f'>{record.name}\n{record.sequence}\n')
    assert not top50_names
    outfp.close()

    outfile2 = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.csv'
    print(outfile2)
    outfp = open(outfile2, 'w', newline='')
    w = csv.writer(outfp)

    for name in foo_df.head(50)['gene'].to_list():
        w.writerow(['0', species, name, "(not reviewed)"])
    outfp.close()
            
    

s__Bariatricus sp004560705
shape: (14, 3)
┌────────────────────────────┬────────────────┬──────┐
│ species                    ┆ gene           ┆ f    │
│ ---                        ┆ ---            ┆ ---  │
│ str                        ┆ str            ┆ f64  │
╞════════════════════════════╪════════════════╪══════╡
│ s__Bariatricus sp004560705 ┆ CHOLCMOC_00863 ┆ 0.97 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_00701 ┆ 0.95 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_02301 ┆ 0.93 │
│ s__Bariatricus sp004560705 ┆ NDCIOGGG_00214 ┆ 0.9  │
│ s__Bariatricus sp004560705 ┆ FMMMFAAO_00746 ┆ 0.89 │
│ …                          ┆ …              ┆ …    │
│ s__Bariatricus sp004560705 ┆ NOACAEKI_00860 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ IEPLHDOE_02248 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ EMNHIHKA_01693 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ ELOEBFJM_01459 ┆ 0.81 │
│ s__Bariatricus sp004560705 ┆ NGBMPMHL_00028 ┆ 0.81 │
└────────────────────────────┴────────────────┴──────┘
../outputs.cds/singlecl

## Explore specific gene/species/metag combinations

In [8]:
depth_df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth')

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64


In [9]:
depth_df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth').filter(pl.col("metag") == "ERR1135199")

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64


## Try something else

In [10]:
names = set()
for filename in glob.glob('../outputs.cds/cds3-genes/*.fa'):
    names.update([ record.name.split(' ')[0] for record in screed.open(filename) ])
names

{'AIEIMHOC_00182',
 'AKLOHKAG_01416',
 'APOLKBKG_00160',
 'BBOFCOCJ_01349',
 'BHHJJAAP_00933',
 'CHOLCMOC_00863',
 'CMHHCIHA_00622',
 'CNENGHLA_00658',
 'CNENGHLA_01260',
 'COCKKAPE_02026',
 'DAPDNANK_01228',
 'DKAPMIEG_00121',
 'EBLLCBKE_00022',
 'EHOAPHDI_01174',
 'FGLPJJIK_03110',
 'FMBBPIHC_01435',
 'GFALEAHI_01277',
 'HJGGINOG_01324',
 'IAFAPFAC_01635',
 'IEPMJGAI_00771',
 'IFIBFMPA_00800',
 'IGEONAAO_00273',
 'JDFEBBFI_00464',
 'KINAFDOL_01550',
 'KPACIEHJ_00390',
 'LEHFNBLN_02301',
 'LJHBCGLM_01294',
 'MMBDDDLD_00384',
 'MNKNIMJG_02022',
 'NADIOELI_00170',
 'NLNOELNI_01902',
 'NMJBBHDE_00624',
 'OBGMJDLM_01221',
 'OIGHMFGK_00345',
 'OMBJBIHD_01711',
 'PKGEEMPI_00758'}

In [11]:
depth_df.filter(pl.col("gene").is_in(names)).sort(by='depth_all')

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64
"""ERR1135199""","""s__Mogibacterium_A kristiansen…","""LJHBCGLM_01294""",393,0,0.0,0.0,NaN
"""ERR1135199""","""s__Mogibacterium_A kristiansen…","""AKLOHKAG_01416""",603,0,0.0,0.0,NaN
"""ERR1135199""","""s__Mogibacterium_A kristiansen…","""BHHJJAAP_00933""",531,0,0.0,0.0,NaN
"""SRR11124687""","""s__Lactobacillus amylovorus""","""OMBJBIHD_01711""",1134,0,0.0,0.0,NaN
"""SRR12795790""","""s__Sodaliphilus sp004557565""","""NLNOELNI_01902""",1968,0,0.0,0.0,NaN
…,…,…,…,…,…,…,…
"""SRR10209683""","""s__Lactobacillus amylovorus""","""OMBJBIHD_01711""",1134,1130,0.996473,673.320106,675.70354
"""SRR10209683""","""s__Lactobacillus amylovorus""","""OBGMJDLM_01221""",2574,2574,1.0,706.990287,706.990287
"""SRR17241521""","""s__Lactobacillus amylovorus""","""BBOFCOCJ_01349""",1134,990,0.873016,2054.869489,2353.759596


In [12]:
depth_df.filter(pl.col("gene").is_in(names)).write_csv('../outputs.cds/cds3-genes/mapping-coverage.csv')